Answer for problem 1: fine-tuning `bert-base-uncased` on Yelp Review Polarity

In [1]:
# !pip -q install -U transformers scikit-learn

In [2]:
import csv
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

pd.set_option("display.max_colwidth", 240)

In [3]:
SEED = 42
MODEL_NAME = "bert-base-uncased"
VAL_SIZE = 0.10
MAX_LENGTH = 256
# TRAIN_BATCH_SIZE = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

RAW_LABEL_TO_ID = {1: 0, 2: 1}
ID_TO_LABEL_NAME = {0: "negative", 1: "positive"}
LABEL_NAME_TO_ID = {"negative": 0, "positive": 1}

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("If GPU not enabled. This *.ipynb will still run, but training will be much slower.")

Using device: cuda


In [4]:
required_files = ["train_0.01.csv", "test_0.01.csv"]
missing_files = [fname for fname in required_files if not Path(fname).exists()]

if missing_files:
    print("Missing files:", ", ".join(missing_files))
    try:
        from google.colab import files
        print("Upload the missing CSV files now.")
        _ = files.upload()
    except ImportError as e:
        raise FileNotFoundError(
            f"Could not find {missing_files}. Put the CSV files in the current working directory."
        ) from e

missing_files = [fname for fname in required_files if not Path(fname).exists()]
if missing_files:
    raise FileNotFoundError(f"Still missing required files: {missing_files}")

train_path = Path("train_0.01.csv")
test_path = Path("test_0.01.csv")
print("Found files:", train_path.resolve(), test_path.resolve())

Found files: /content/train_0.01.csv /content/test_0.01.csv


In [5]:
train_df = pd.read_csv(
    train_path,
    header=None,
    names=["raw_label", "text"],
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
)

test_df = pd.read_csv(
    test_path,
    header=None,
    names=["raw_label", "text"],
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
)

for df_name, df in [("train", train_df), ("test", test_df)]:
    if df["raw_label"].isna().any() or df["text"].isna().any():
        raise ValueError(f"{df_name} contains missing labels or text entries.")
    df["label"] = df["raw_label"].map(RAW_LABEL_TO_ID)
    if df["label"].isna().any():
        bad = sorted(df.loc[df["label"].isna(), "raw_label"].unique().tolist())
        raise ValueError(f"Unexpected labels found in {df_name}: {bad}")
    df["label"] = df["label"].astype(int)
    df["text"] = df["text"].astype(str)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
train_df.head()

Train shape: (5600, 3)
Test shape:  (380, 3)


,raw_label,text,label
0,1,"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually...",0
1,2,"Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff...",1
2,1,"I don't know what Dr. Goldberg was like before moving to Arizona, but let me tell you, STAY AWAY from this doctor and this office. I was going to Dr. Johnson before he left and Goldberg took over when Johnson left. He is not a caring d...",0
3,1,"I'm writing this review to give you a heads up before you see this Doctor. The office staff and administration are very unprofessional. I left a message with multiple people regarding my bill, and no one ever called me back. I had to ho...",0
4,2,"All the food is great here. But the best thing they have is their wings. Their wings are simply fantastic!! The \""Wet Cajun\"" are by the best & most popular. I also like the seasoned salt wings. Wing Night is Monday & Wednesday night...",1


In [6]:
def summarize_split(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "split": [name],
            "rows": [len(df)],
            "negative_count": [(df["label"] == 0).sum()],
            "positive_count": [(df["label"] == 1).sum()],
            "avg_chars": [df["text"].str.len().mean()],
            "median_chars": [df["text"].str.len().median()],
        }
    )

overview_df = pd.concat(
    [
        summarize_split(train_df, "train_full"),
        summarize_split(test_df, "test"),
    ],
    ignore_index=True,
)

display(overview_df)

label_note = pd.DataFrame(
    {
        "raw_label_in_csv": [1, 2],
        "meaning": ["negative", "positive"],
        "model_label_id": [0, 1],
    }
)
display(label_note)

,split,rows,negative_count,positive_count,avg_chars,median_chars
0,train_full,5600,2919,2681,733.154464,549.0
1,test,380,181,199,740.742105,575.0


,raw_label_in_csv,meaning,model_label_id
0,1,negative,0
1,2,positive,1


In [7]:
train_split_df, val_df = train_test_split(
    train_df[["text", "label"]].copy(),
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["label"],
)

train_split_df = train_split_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_eval_df = test_df[["text", "label"]].copy().reset_index(drop=True)

split_sizes_df = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_split_df), len(val_df), len(test_eval_df)],
        "negative_count": [
            (train_split_df["label"] == 0).sum(),
            (val_df["label"] == 0).sum(),
            (test_eval_df["label"] == 0).sum(),
        ],
        "positive_count": [
            (train_split_df["label"] == 1).sum(),
            (val_df["label"] == 1).sum(),
            (test_eval_df["label"] == 1).sum(),
        ],
    }
)

display(split_sizes_df)

,split,rows,negative_count,positive_count
0,train,5040,2627,2413
1,validation,560,292,268
2,test,380,181,199


In [8]:
hyperparams_df = pd.DataFrame(
    {
        "hyperparameter": [
            "model_name",
            "validation_fraction",
            "max_length",
            "train_batch_size",
            "eval_batch_size",
            "learning_rate",
            "weight_decay",
            "num_epochs",
            "warmup_ratio",
            "gradient_clip_norm",
            "optimizer",
            "early_stopping_patience",
            "random_seed",
        ],
        "value": [
            MODEL_NAME,
            VAL_SIZE,
            MAX_LENGTH,
            TRAIN_BATCH_SIZE,
            EVAL_BATCH_SIZE,
            LEARNING_RATE,
            WEIGHT_DECAY,
            NUM_EPOCHS,
            WARMUP_RATIO,
            GRAD_CLIP_NORM,
            "AdamW",
            EARLY_STOPPING_PATIENCE,
            SEED,
        ],
    }
)

display(hyperparams_df)

,hyperparameter,value
0,model_name,bert-base-uncased
1,validation_fraction,0.1
2,max_length,256
3,train_batch_size,16
4,eval_batch_size,32
5,learning_rate,0.00002
6,weight_decay,0.01
7,num_epochs,3
8,warmup_ratio,0.1
9,gradient_clip_norm,1.0


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in self.encodings}
        item["labels"] = int(self.labels[idx])
        return item

train_dataset = ReviewDataset(
    train_split_df["text"].tolist(),
    train_split_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = ReviewDataset(
    val_df["text"].tolist(),
    val_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = ReviewDataset(
    test_eval_df["text"].tolist(),
    test_eval_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_workers = 2 if device.type == "cuda" else 0
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train batches: 315
Validation batches: 18
Test batches: 12


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID_TO_LABEL_NAME,
    label2id=LABEL_NAME_TO_ID,
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_training_steps = len(train_loader) * NUM_EPOCHS
num_warmup_steps = int(WARMUP_RATIO * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_training_steps,
)

BEST_STATE_PATH = Path("best_bert_yelp_state.pt")
BEST_EXPORT_DIR = Path("bert_yelp_sentiment_best")

print(f"Total training steps: {total_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total training steps: 945
Warmup steps: 94


In [11]:
def compute_binary_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    return {
        "accuracy": float(acc),
        "precision_positive": float(precision),
        "recall_positive": float(recall),
        "f1_positive": float(f1),
    }

@torch.no_grad()
def predict_with_probs(model, dataloader, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_pos_probs = []
    all_confidences = []

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

        all_labels.extend(batch["labels"].cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_pos_probs.extend(probs[:, 1].cpu().numpy().tolist())
        all_confidences.extend(probs.max(dim=-1).values.cpu().numpy().tolist())

    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_pos_probs),
        np.array(all_confidences),
    )

def evaluate_model(model, dataloader, device):
    y_true, y_pred, pos_probs, confidences = predict_with_probs(model, dataloader, device)
    metrics = compute_binary_metrics(y_true, y_pred)
    return metrics, y_true, y_pred, pos_probs, confidences

In [12]:
history = []
best_val_f1 = -1.0
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = running_loss / max(len(train_loader), 1)

    val_metrics, _, _, _, _ = evaluate_model(model, val_loader, device)
    epoch_record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(epoch_record)

    print(
        f"Epoch {epoch}: "
        f"train_loss={avg_train_loss:.4f}, "
        f"val_accuracy={val_metrics['accuracy']:.4f}, "
        f"val_precision={val_metrics['precision_positive']:.4f}, "
        f"val_recall={val_metrics['recall_positive']:.4f}, "
        f"val_f1={val_metrics['f1_positive']:.4f}"
    )

    if val_metrics["f1_positive"] > best_val_f1:
        best_val_f1 = val_metrics["f1_positive"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_STATE_PATH)
        print(f"  -> Saved new best model to {BEST_STATE_PATH}")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement. Patience counter: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

if not BEST_STATE_PATH.exists():
    raise FileNotFoundError("Best model checkpoint was not saved correctly.")

model.load_state_dict(torch.load(BEST_STATE_PATH, map_location=device))
model.to(device)

history_df = pd.DataFrame(history)
display(history_df)

Epoch 1/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 1: train_loss=0.3389, val_accuracy=0.9518, val_precision=0.9382, val_recall=0.9627, val_f1=0.9503
  -> Saved new best model to best_bert_yelp_state.pt


Epoch 2/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

...

Epoch 2: train_loss=0.1326, val_accuracy=0.9554, val_precision=0.9655, val_recall=0.9403, val_f1=0.9527
  -> Saved new best model to best_bert_yelp_state.pt


Epoch 3/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 3: train_loss=0.0573, val_accuracy=0.9589, val_precision=0.9554, val_recall=0.9590, val_f1=0.9572
  -> Saved new best model to best_bert_yelp_state.pt


,epoch,train_loss,val_accuracy,val_precision_positive,val_recall_positive,val_f1_positive
0,1,0.338852,0.951786,0.938182,0.962687,0.950276
1,2,0.132603,0.955357,0.965517,0.940299,0.952741
2,3,0.057255,0.958929,0.955390,0.958955,0.957169


In [13]:
test_metrics, test_y_true, test_y_pred, test_pos_probs, test_confidences = evaluate_model(
    model, test_loader, device
)

metrics_table = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Precision (positive class)",
            "Recall (positive class)",
            "F1 (positive class)",
        ],
        "value": [
            test_metrics["accuracy"],
            test_metrics["precision_positive"],
            test_metrics["recall_positive"],
            test_metrics["f1_positive"],
        ],
    }
)

metrics_table["value"] = metrics_table["value"].map(lambda x: round(float(x), 4))
display(metrics_table)

print(
    f"Test accuracy = {test_metrics['accuracy']:.4f}\n"
    f"Test precision (positive) = {test_metrics['precision_positive']:.4f}\n"
    f"Test recall (positive) = {test_metrics['recall_positive']:.4f}\n"
    f"Test F1 (positive) = {test_metrics['f1_positive']:.4f}"
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,metric,value
0,Accuracy,0.9553
1,Precision (positive class),0.9596
2,Recall (positive class),0.9548
3,F1 (positive class),0.9572


Test accuracy = 0.9553
Test precision (positive) = 0.9596
Test recall (positive) = 0.9548
Test F1 (positive) = 0.9572


In [14]:
BEST_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(BEST_EXPORT_DIR)
tokenizer.save_pretrained(BEST_EXPORT_DIR)
print(f"Saved best model + tokenizer to: {BEST_EXPORT_DIR.resolve()}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model + tokenizer to: /content/bert_yelp_sentiment_best


In [15]:
def truncate_text(text: str, max_chars: int = 220) -> str:
    text = " ".join(str(text).split())
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."

def pick_diverse_examples(df: pd.DataFrame, group_col: str, max_total: int = 5, base_per_group: int = 2) -> pd.DataFrame:
    picked_parts = []
    picked_indices = []

    for group_value in sorted(df[group_col].dropna().unique().tolist()):
        group_df = df[df[group_col] == group_value].sort_values("prediction_confidence", ascending=False)
        take = group_df.head(base_per_group)
        if not take.empty:
            picked_parts.append(take)
            picked_indices.extend(take.index.tolist())

    combined = pd.concat(picked_parts, axis=0) if picked_parts else df.head(0).copy()

    if len(combined) < max_total:
        remaining = df.drop(index=picked_indices, errors="ignore").sort_values(
            "prediction_confidence", ascending=False
        )
        combined = pd.concat([combined, remaining.head(max_total - len(combined))], axis=0)

    combined = combined.sort_values("prediction_confidence", ascending=False).head(max_total).copy()
    return combined

analysis_df = test_eval_df.copy().reset_index(drop=True)
analysis_df["true_label_id"] = test_y_true
analysis_df["predicted_label_id"] = test_y_pred
analysis_df["true_label"] = analysis_df["true_label_id"].map(ID_TO_LABEL_NAME)
analysis_df["predicted_label"] = analysis_df["predicted_label_id"].map(ID_TO_LABEL_NAME)
analysis_df["positive_probability"] = test_pos_probs
analysis_df["prediction_confidence"] = test_confidences
analysis_df["correct"] = analysis_df["true_label_id"] == analysis_df["predicted_label_id"]
analysis_df["text_truncated"] = analysis_df["text"].map(truncate_text)
analysis_df["error_type"] = np.where(
    analysis_df["correct"],
    "correct",
    np.where(
        (analysis_df["true_label_id"] == 1) & (analysis_df["predicted_label_id"] == 0),
        "false_negative",
        "false_positive",
    ),
)

full_test_token_lengths = tokenizer(
    analysis_df["text"].tolist(),
    truncation=False,
    add_special_tokens=True,
)["input_ids"]
analysis_df["token_length_full"] = [len(ids) for ids in full_test_token_lengths]
analysis_df["was_truncated_at_max_length"] = analysis_df["token_length_full"] > MAX_LENGTH

correct_examples = pick_diverse_examples(
    analysis_df[analysis_df["correct"]],
    group_col="true_label",
    max_total=5,
    base_per_group=2,
)[
    ["text_truncated", "true_label", "predicted_label", "positive_probability", "prediction_confidence"]
].copy()

error_examples = pick_diverse_examples(
    analysis_df[~analysis_df["correct"]],
    group_col="error_type",
    max_total=5,
    base_per_group=2,
)[
    ["text_truncated", "true_label", "predicted_label", "positive_probability", "prediction_confidence"]
].copy()

for df_ in [correct_examples, error_examples]:
    for col in ["positive_probability", "prediction_confidence"]:
        if col in df_.columns:
            df_[col] = df_[col].map(lambda x: round(float(x), 4))

print("Correctly classified review examples:")
display(correct_examples)

print("Misclassified review examples:")
display(error_examples)

print(f"Total test errors: {(~analysis_df['correct']).sum()} out of {len(analysis_df)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (575 > 512). Running this sequence through the model will result in indexing errors


Correctly classified review examples:


,text_truncated,true_label,predicted_label,positive_probability,prediction_confidence
191,I surmised two things from my visit to Thai Cuisine:\n\na) the competition among thai restaurants must not be very stiff in this town\n\nb) yinzers wouldn't know good thai food if it flew into their open mouths\n\nThi...,negative,negative,0.0008,0.9992
231,"Harris Grill was long one of my favorite spots to goto just about any night of the week. However it's changed in recent months and I don't think I will return.\n\nOn a recent Thursday night, my friend and I had possib...",negative,negative,0.0008,0.9992
271,Will never come here again!\n\nMy cat has cancer and was experiencing some new symptoms that needed possible ultrasounds. We came to this CVS location because it was after hours and the CVS I usually go to in Matthews...,negative,negative,0.0008,0.9992
313,This is a great stop for a meal if your going to a Bobcat's game or any event at Time Warner arena.\nThey have covered garage parking for just $5 and a menu that just won't quit. They've got something for everyone.\nT...,positive,positive,0.9988,0.9988
311,WOW! Very good food!!!!! The wait staff was very friendly and quick! I always love the sharing family style of chinese restaurants! You must go here!!! AWESOME PLACE!!!!! And the person who suggested the donuts!!! THA...,positive,positive,0.9988,0.9988


Misclassified review examples:


,text_truncated,true_label,predicted_label,positive_probability,prediction_confidence
336,"I've gotten massages from Modern Salon several times, and loved them all there... until the last time, when I lost my penchant- after learning the masseuse I had used before, no longer worked there. His replacement wa...",positive,negative,0.0013,0.9987
264,Says they deliver on here... Wrong & wrong again & should not be checked! I like Applebee's & thought the delivery was something new for the Pittsburgh area... Don't know if this is something Yelp does or something so...,positive,negative,0.0020,0.9980
201,"I stopped at Rohrich Toyota to look at their selection of used vehicles. I was quickly greeted by Paul - who was great! I test drove a Honda Element, which was clean and comfortable to drive, but the brakes on this on...",negative,positive,0.9975,0.9975
308,Ri Ra is one of the coolest bars in Charlotte from a decor/ambiance perspective. After work it's loud and crowded and energetic - exactly as it should be. \n\nWhat I really wish is that the kitchen staff were able to ...,negative,positive,0.9969,0.9969
270,"Get your wallet ready, the prices are crazy high!",negative,positive,0.9960,0.9960


Total test errors: 17 out of 380
